# Hysteresis Analysis: Effect of MD1 Conditioning on SFTPRO & LHC

**Campaign:** MBB max speed + NMR (2026-03-06)

| Session | MD1 type (x20) | Peak I during MD1 | Then |
|---------|---------------|-------------------|------|
| **200 GeV** | Full MD1 cycle | ~2267 A | SFTPRO -> LHC |
| **26 GeV** | Flattened MD1 | ~301 A (injection only) | SFTPRO -> LHC |

**Goal:** Quantify hysteresis difference on SFTPRO and LHC plateaus caused by different MD1 conditioning.

**Key current levels:**
- Idle: ~155 A
- Injection: ~301 A (not present in SFTPRO)
- SFTPRO top: ~4816 A
- LHC top: ~5781 A

**Encoder offset correction:** Raw C_1 sits at -90.4 deg (encoder zero ~90 deg from field axis).
The `_wrap_arg_to_pm_pi_over_2` boundary at -pi/2 causes B1 to come out negative.
Correction equivalent to `encoder_offset_rad = -pi/2` is applied after loading:
B1 flips positive, even-order b_n flip sign, odd-order and all |b_n| unchanged.

**NMR:** Available at top plateaus; reliable values >= 1 s after lock.

---
## 1. Configuration & Imports

In [ ]:
# === CONFIGURATION ===
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

%matplotlib widget
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 100,
})

REPO_ROOT = Path(".").resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "pyproject.toml").exists() or (REPO_ROOT / ".git").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
MEAS = REPO_ROOT / "measurements"

SESSIONS = {
    "200 GeV": {
        "desc": "Full MD1 (peak ~2267 A)",
        "results_body": "MBB/2026-03-06_max_speed_NMR/20260306_152236_SPS_MBB/20260306_152257_MBB/20260306_152257_MBB_Run_00_I_100.00A_body_results.txt",
        "nmr_h5": "MBB/2026-03-06_max_speed_NMR/20260306_152447_TestCaylarTeslameterNMR20_ md1full_.h5",
    },
    "26 GeV": {
        "desc": "Flattened MD1 (injection only, ~301 A)",
        "results_body": "MBB/2026-03-06_max_speed_NMR/20260306_153553_SPS_MBB/20260306_153614_MBB/20260306_153614_MBB_Run_00_I_100.00A_body_results.txt",
        "nmr_h5": "MBB/2026-03-06_max_speed_NMR/20260306_153650_TestCaylarTeslameterNMR20_ md1flat_.h5",
    },
}
SESSION_NAMES = list(SESSIONS.keys())

# Encoder offset: raw arg(C_1) = -90.4 deg, encoder zero ~90 deg from field axis
ENCODER_OFFSET_RAD = -np.pi / 2  # for documentation; correction applied post-hoc below

# Current bands for plateau classification (A)
I_BANDS = {
    "idle":       (145, 170),
    "injection":  (290, 315),
    "sftpro_top": (4800, 4830),
    "lhc_top":    (5770, 5790),
}

# Settling: use last N turns of each plateau for statistics
N_SETTLE = {"idle": 18, "injection": 18, "sftpro_top": 15, "lhc_top": 5}

# NMR: discard first N seconds after lock
NMR_SETTLE_S = 1.0

# Hall-probe thresholds to identify NMR lock periods
HALL_SFTPRO_MIN = 1.6   # T
HALL_LHC_MIN    = 1.95  # T

# Colours
COLORS = {"200 GeV": "tab:blue", "26 GeV": "tab:orange"}
PLATEAU_COLORS = {
    "idle": "tab:cyan", "injection": "tab:green",
    "sftpro_top": "tab:red", "lhc_top": "tab:purple",
}

print("Hysteresis Analysis: Effect of MD1 Conditioning on SFTPRO & LHC")
print("=" * 70)
for name, cfg in SESSIONS.items():
    print(f"  {name}: {cfg['desc']}")

---
## 2. Data Loading

In [ ]:
# Load per-turn results (body segment) for both sessions
turns = {}
for name, cfg in SESSIONS.items():
    fpath = MEAS / cfg["results_body"]
    assert fpath.exists(), f"Missing: {fpath}"
    df = pd.read_csv(fpath, sep="\t")
    df = df.rename(columns={
        "Time(s)": "t_s", "Duration(s)": "dur_s", "I(A)": "I_A",
        "Ramprate(A/s)": "rr",
        "B_main(T)": "B1_T", "A_main(T)": "A1_T",
        "b2(Units)": "b2", "a2(Units)": "a2",
        "b3(Units)": "b3", "a3(Units)": "a3",
        "b5(Units)": "b5", "a5(Units)": "a5",
    })
    df["turn"] = np.arange(len(df))
    df["t_end"] = df["t_s"] + df["dur_s"]

    # --- Encoder offset correction (equivalent to encoder_offset_rad = -pi/2) ---
    # Raw arg(C_1) = -90.4 deg sits 0.4 deg past the _wrap_arg_to_pm_pi_over_2
    # boundary at -pi/2, causing B1 to come out negative.
    # Correction: B_main flips sign; normalised b_n, a_n multiply by (-1)^(n-1)
    #   -> even n: sign flip;  odd n: unchanged.  |b_n| invariant for all n.
    df["B1_T"] = -df["B1_T"]
    if "A1_T" in df.columns:
        df["A1_T"] = -df["A1_T"]
    for n in range(2, 16):
        if n % 2 == 0:  # even order: flip sign
            for prefix in ["b", "a"]:
                for col in [f"{prefix}{n}", f"{prefix}{n}(Units)"]:
                    if col in df.columns:
                        df[col] = -df[col]

    turns[name] = df
    print(f"{name}: {len(df)} turns, B1 range {df.B1_T.min():.4f} to {df.B1_T.max():.4f} T "
          f"(positive \u2714)")

In [ ]:
# Load NMR data
nmr_data = {}
for name, cfg in SESSIONS.items():
    fpath = MEAS / cfg["nmr_h5"]
    assert fpath.exists(), f"Missing: {fpath}"
    with h5py.File(fpath, "r") as f:
        data = f["RawData/SFTPRO-NMR"][:]
        start_s = f["RawData/CaylarStartTime(s)"][0, 0]
    nmr_data[name] = {
        "t_ms": data[:, 0], "nmr": data[:, 1],
        "hall": data[:, 2], "lock": data[:, 3].astype(int),
        "start_s": start_s,
    }
    n_locked = int(data[:, 3].sum())
    print(f"{name} NMR: {len(data)} pts, {n_locked} locked, "
          f"Hall {data[:,2].min():.3f}\u2013{data[:,2].max():.3f} T")

---
## 3. Current Profile & Cycle Structure

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

for ax, name in zip(axes, SESSION_NAMES):
    df = turns[name]
    ax.plot(df.t_s, df.I_A, "-", linewidth=0.5, color=COLORS[name])
    ax.set_ylabel("I (A)")
    ax.set_title(f"{name} \u2014 {SESSIONS[name]['desc']}")

    for bname, (lo, hi) in I_BANDS.items():
        ax.axhspan(lo, hi, alpha=0.08, color="grey")
        ax.text(df.t_s.iloc[0] + 2, (lo + hi) / 2, bname, fontsize=7,
                va="center", color="grey")

    high_mask = df.I_A > 4000
    if high_mask.any():
        t_sft = df.t_s[high_mask].iloc[0]
        ax.axvline(t_sft - 5, color="red", ls="--", lw=0.8, alpha=0.6)
        ax.text(t_sft - 6, df.I_A.max() * 0.5, "\u2190 MD1 region | SFTPRO+LHC \u2192",
                fontsize=8, color="red", ha="center", rotation=90)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Current Profile \u2014 Both Sessions", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Plateau Detection & Classification

In [ ]:
RAMPRATE_THRESHOLD = 5.0  # A/s — plateau noise floor ~2.5 A/s peak, ramps jump to 100+ A/s

# Skip first N MD1 cycles in convergence plot (first cycle is from unknown state)
MD1_SKIP_FIRST = 1


def classify_and_group(df, I_bands, rr_thresh, min_consecutive=3):
    """Classify turns into current bands (plateau only if |rr| < threshold)."""
    I = df["I_A"].values
    rr = df["rr"].values
    n = len(I)
    band = np.full(n, "", dtype=object)

    on_plateau = np.abs(rr) < rr_thresh
    for bname, (lo, hi) in I_bands.items():
        mask = (I >= lo) & (I <= hi) & on_plateau
        band[mask] = bname

    groups = []
    i = 0
    while i < n:
        if band[i] != "":
            label = band[i]
            start = i
            while i < n and band[i] == label:
                i += 1
            length = i - start
            if length >= min_consecutive:
                groups.append({
                    "label": label, "start": start, "end": i,
                    "n_turns": length,
                    "I_mean": I[start:i].mean(),
                    "t_start": df["t_s"].iloc[start],
                    "t_end": df["t_s"].iloc[i - 1],
                })
        else:
            i += 1
    return groups


def assign_cycles(groups):
    """Assign each group to a cycle based on temporal position."""
    sft = next((g for g in groups if g["label"] == "sftpro_top"), None)
    lhc = next((g for g in groups if g["label"] == "lhc_top"), None)
    t_sft = sft["t_start"] if sft else 1e9
    t_lhc = lhc["t_start"] if lhc else 1e9

    md1_inj_idx = 0
    for g in groups:
        t = g["t_start"]
        lab = g["label"]
        if t < t_sft:
            if lab == "injection":
                md1_inj_idx += 1
                g["cycle"] = "MD1"
                g["cycle_idx"] = md1_inj_idx
            elif lab == "idle":
                g["cycle"] = "pre-SFTPRO"
                g["cycle_idx"] = 0
            else:
                g["cycle"] = "MD1"
                g["cycle_idx"] = 0
        elif lab == "sftpro_top":
            g["cycle"] = "SFTPRO"
            g["cycle_idx"] = 0
        elif t > t_sft and t < t_lhc:
            if lab == "idle":
                g["cycle"] = "SFTPRO\u2192LHC"
            elif lab == "injection":
                g["cycle"] = "LHC"
            else:
                g["cycle"] = "transition"
            g["cycle_idx"] = 0
        elif lab == "lhc_top":
            g["cycle"] = "LHC"
            g["cycle_idx"] = 0
        else:
            g["cycle"] = "post-LHC"
            g["cycle_idx"] = 0
    return groups


# Run classification for both sessions
all_groups = {}
for name in SESSION_NAMES:
    grps = classify_and_group(turns[name], I_BANDS, RAMPRATE_THRESHOLD)
    grps = assign_cycles(grps)
    all_groups[name] = grps

    print(f"\n=== {name} ===")
    for g in grps:
        settle = min(N_SETTLE.get(g["label"], g["n_turns"]), g["n_turns"])
        print(f"  {g['cycle']:12s} | {g['label']:12s} | turns {g['start']:5d}\u2013{g['end']:5d} "
              f"({g['n_turns']:4d} turns) | I={g['I_mean']:.0f} A | settle last {settle}")

---
## 5. Turn Map on DCCT Current Profile

Each turn drawn as a horizontal bar at its mean current.
- **Green:** settled turns (used for averaging)
- **Orange:** on plateau but excluded (eddy settling)
- **Red:** ramp turns (not on any plateau)

In [ ]:
def classify_turns(df, grps, n_settle_dict):
    """Return per-turn status array: 'ramp', 'transient', or 'settled'."""
    status = np.full(len(df), "ramp", dtype=object)
    for g in grps:
        n_s = min(n_settle_dict.get(g["label"], g["n_turns"]), g["n_turns"])
        settle_start = g["end"] - n_s
        status[g["start"]:settle_start] = "transient"
        status[settle_start:g["end"]] = "settled"
    return status


COLOR_MAP = {"ramp": "tab:red", "transient": "tab:orange", "settled": "tab:green"}
LABEL_MAP = {"ramp": "Ramp", "transient": "Plateau (excluded)",
             "settled": "Plateau (settled, used)"}


def draw_turn_bars(ax, df, status, idx=None):
    """Draw horizontal bars for each turn, coloured by status."""
    if idx is None:
        idx = range(len(df))
    drawn = set()
    for i in idx:
        s = status[i]
        kw = {}
        if s not in drawn:
            kw["label"] = LABEL_MAP[s]
            drawn.add(s)
        ax.plot([df.t_s.iloc[i], df.t_end.iloc[i]],
                [df.I_A.iloc[i], df.I_A.iloc[i]],
                linewidth=2.5, color=COLOR_MAP[s], alpha=0.85,
                solid_capstyle="butt", zorder=2, **kw)


# --- Full current profile ---
fig, axes = plt.subplots(len(SESSION_NAMES), 1, figsize=(16, 5 * len(SESSION_NAMES)),
                         sharex=False)
if len(SESSION_NAMES) == 1:
    axes = [axes]

for ax, name in zip(axes, SESSION_NAMES):
    df = turns[name]
    status = classify_turns(df, all_groups[name], N_SETTLE)
    draw_turn_bars(ax, df, status)
    ax.set_ylabel("I (A)")
    ax.set_title(f"{name} \u2014 {SESSIONS[name]['desc']}")
    ax.legend(loc="upper right", fontsize=9)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Current Profile with Turn Classification", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- Zoom on SFTPRO + LHC region ---
fig, axes = plt.subplots(len(SESSION_NAMES), 1, figsize=(16, 5 * len(SESSION_NAMES)),
                         sharex=False)
if len(SESSION_NAMES) == 1:
    axes = [axes]

for ax, name in zip(axes, SESSION_NAMES):
    df = turns[name]
    grps = all_groups[name]
    status = classify_turns(df, grps, N_SETTLE)

    non_md1 = [g for g in grps if g["cycle"] not in ("MD1",)]
    if not non_md1:
        continue
    t_lo = non_md1[0]["t_start"] - 10
    t_hi = df.t_s.iloc[-1] + 2
    idx_zoom = np.where((df.t_s.values >= t_lo) & (df.t_s.values <= t_hi))[0]

    draw_turn_bars(ax, df, status, idx_zoom)
    ax.set_ylabel("I (A)")
    ax.set_title(f"{name} \u2014 SFTPRO + LHC region")
    ax.legend(loc="upper right", fontsize=9)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("SFTPRO + LHC Region \u2014 Turn Classification (zoom)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 5b. Transfer Function (settled turns only)

TF = B1 / I (T/kA) for all settled (green) turns, plotted turn-by-turn and vs current.

In [ ]:
# Transfer function for all settled turns
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name in SESSION_NAMES:
    df = turns[name]
    grps = all_groups[name]
    status = classify_turns(df, grps, N_SETTLE)
    mask = status == "settled"
    sub = df[mask].copy()

    # Exclude idle turns (I < 100 A) — TF = B1/I diverges at low I
    sub = sub[sub.I_A > 100]
    sub["TF"] = sub.B1_T / sub.I_A * 1000  # T/kA

    # Left: TF vs time (turn-by-turn)
    axes[0].plot(sub.t_s, sub.TF, ".", ms=4, color=COLORS[name], alpha=0.6, label=name)

    # Right: TF vs I (hysteresis loop)
    axes[1].plot(sub.I_A, sub.TF, ".", ms=4, color=COLORS[name], alpha=0.6, label=name)

axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("TF = B1/I (T/kA)")
axes[0].set_title("Transfer Function vs Time (settled turns)")
axes[0].legend(fontsize=9)

axes[1].set_xlabel("I (A)")
axes[1].set_ylabel("TF = B1/I (T/kA)")
axes[1].set_title("Transfer Function vs Current (settled turns)")
axes[1].legend(fontsize=9)

fig.suptitle("Transfer Function \u2014 Settled Turns Only (body)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Print TF at each plateau level
print("TF at plateau levels (settled turns):")
for name in SESSION_NAMES:
    df = turns[name]
    status = classify_turns(df, all_groups[name], N_SETTLE)
    mask = status == "settled"
    sub = df[mask]
    print(f"\n  {name}:")
    for label in ["injection", "sftpro_top", "lhc_top"]:
        lo, hi = I_BANDS[label]
        plat = sub[(sub.I_A >= lo) & (sub.I_A <= hi)]
        if len(plat) > 0:
            tf = plat.B1_T / plat.I_A * 1000
            print(f"    {label:12s}: I={plat.I_A.mean():.0f} A, TF={tf.mean():.4f} \u00b1 {tf.std():.4f} T/kA ({len(plat)} turns)")

---
## 6. MD1 Minor Loop Convergence

20 MD1 cycles played to reach a repeatable minor loop.
Track per-cycle averages (last 18 settled turns) of B1, b2, b3 at injection (~301 A).

In [ ]:
def settled_stats(df, group, n_settle):
    """Return mean and std of settled turns for key quantities."""
    n = min(n_settle, group["n_turns"])
    sl = slice(group["end"] - n, group["end"])
    sub = df.iloc[sl]
    out = {"n_turns": n}
    for col in ["I_A", "B1_T", "b2", "b3"]:
        if col in sub.columns:
            out[f"{col}_mean"] = sub[col].mean()
            out[f"{col}_std"]  = sub[col].std()
    return out


# Extract MD1 injection convergence
md1_conv = {}
for name in SESSION_NAMES:
    md1_grps = [g for g in all_groups[name] if g["cycle"] == "MD1" and g["label"] == "injection"]
    rows = []
    for g in md1_grps:
        s = settled_stats(turns[name], g, N_SETTLE["injection"])
        s["cycle_idx"] = g["cycle_idx"]
        rows.append(s)
    md1_conv[name] = pd.DataFrame(rows)
    print(f"{name}: {len(rows)} MD1 injection plateaus")


# Plot convergence (skip first MD1_SKIP_FIRST cycles — first cycle from unknown state)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for col, ylabel, ax in zip(["B1_T", "b2", "b3"],
                            ["B1 (T)", "b2 (units)", "b3 (units)"],
                            axes):
    for name in SESSION_NAMES:
        df_c = md1_conv[name]
        df_c = df_c[df_c.cycle_idx > MD1_SKIP_FIRST]
        ax.errorbar(df_c.cycle_idx, df_c[f"{col}_mean"], yerr=df_c[f"{col}_std"],
                    fmt="o-", ms=4, capsize=2, color=COLORS[name], alpha=0.8, label=name)
    ax.set_xlabel("MD1 cycle #")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} at injection (~301 A)")
    if col != "B1_T":
        ax.axhline(0, color="grey", lw=0.5)
    ax.legend(fontsize=9)

fig.suptitle("MD1 Minor Loop Convergence \u2014 Injection Plateau (body, cycle 1 excluded)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Convergence check
print("\nConvergence check (first 5 vs last 5 of plotted MD1 cycles):")
for name in SESSION_NAMES:
    df_c = md1_conv[name]
    df_c = df_c[df_c.cycle_idx > MD1_SKIP_FIRST]
    for col, unit in [("B1_T", "T"), ("b2", "units"), ("b3", "units")]:
        first5 = df_c[f"{col}_mean"].iloc[:5].mean()
        last5  = df_c[f"{col}_mean"].iloc[-5:].mean()
        diff = last5 - first5
        print(f"  {name} {col}: first5={first5:.6f}, last5={last5:.6f}, \u0394={diff:.6f} {unit}")

---
## 7. SFTPRO Hysteresis

Compare SFTPRO plateau values between sessions.
- **Top** (~4816 A): direct hysteresis effect of MD1 conditioning
- **Idle** (~155 A): post-SFTPRO idle (long, well-settled)

Note: SFTPRO has no injection plateau.

In [ ]:
def get_group(groups, cycle, label):
    """Find the first group matching cycle and label."""
    for g in groups:
        if g["cycle"] == cycle and g["label"] == label:
            return g
    return None


def get_settled_slice(df, group, n_settle):
    """Return the settled sub-DataFrame."""
    n = min(n_settle, group["n_turns"])
    return df.iloc[group["end"] - n : group["end"]]


# --- SFTPRO top ---
print("=== SFTPRO Top (~4816 A) ===")
sft_stats = {}
for name in SESSION_NAMES:
    g = get_group(all_groups[name], "SFTPRO", "sftpro_top")
    sub = get_settled_slice(turns[name], g, N_SETTLE["sftpro_top"])
    sft_stats[name] = sub
    print(f"  {name}: {len(sub)} settled turns, I={sub.I_A.mean():.1f} A, "
          f"B1={sub.B1_T.mean():.6f} T, b2={sub.b2.mean():.2f}, b3={sub.b3.mean():.2f}")

delta_B1 = sft_stats["26 GeV"].B1_T.mean() - sft_stats["200 GeV"].B1_T.mean()
delta_b2 = sft_stats["26 GeV"].b2.mean() - sft_stats["200 GeV"].b2.mean()
delta_b3 = sft_stats["26 GeV"].b3.mean() - sft_stats["200 GeV"].b3.mean()
print(f"\n  \u0394(26-200): B1 = {delta_B1*1e6:.1f} \u00b5T, b2 = {delta_b2:.3f} units, b3 = {delta_b3:.3f} units")

# --- SFTPRO top turn-by-turn overlay ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for col, ylabel, ax in zip(["B1_T", "b2", "b3"],
                            ["B1 (T)", "b2 (units)", "b3 (units)"], axes):
    for name in SESSION_NAMES:
        g = get_group(all_groups[name], "SFTPRO", "sftpro_top")
        sub = turns[name].iloc[g["start"]:g["end"]]
        local_t = sub.t_s.values - sub.t_s.values[0]
        ax.plot(local_t, sub[col], "o-", ms=3, color=COLORS[name], alpha=0.7, label=name)
    ax.set_xlabel("Time on plateau (s)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"SFTPRO top \u2014 {ylabel}")
    ax.legend(fontsize=9)

fig.suptitle("SFTPRO Top Plateau \u2014 Turn-by-Turn (body)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Post-SFTPRO idle ---
print("=== Post-SFTPRO Idle (~155 A, between SFTPRO and LHC) ===")
idle_stats = {}
for name in SESSION_NAMES:
    g = get_group(all_groups[name], "SFTPRO\u2192LHC", "idle")
    if g is None:
        print(f"  {name}: no post-SFTPRO idle found")
        continue
    sub = get_settled_slice(turns[name], g, N_SETTLE["idle"])
    idle_stats[name] = sub
    print(f"  {name}: {g['n_turns']} total turns, {len(sub)} settled, I={sub.I_A.mean():.1f} A, "
          f"B1={sub.B1_T.mean():.6f} T, b2={sub.b2.mean():.2f}, b3={sub.b3.mean():.2f}")

if len(idle_stats) == 2:
    delta_B1 = idle_stats["26 GeV"].B1_T.mean() - idle_stats["200 GeV"].B1_T.mean()
    print(f"\n  \u0394(26-200): B1 = {delta_B1*1e6:.1f} \u00b5T")

# Turn-by-turn idle overlay
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for col, ylabel, ax in zip(["B1_T", "b2", "b3"],
                            ["B1 (T)", "b2 (units)", "b3 (units)"], axes):
    for name in SESSION_NAMES:
        g = get_group(all_groups[name], "SFTPRO\u2192LHC", "idle")
        if g is None:
            continue
        sub = turns[name].iloc[g["start"]:g["end"]]
        local_t = sub.t_s.values - sub.t_s.values[0]
        ax.plot(local_t, sub[col], "o-", ms=2, color=COLORS[name], alpha=0.7, label=name)
    ax.set_xlabel("Time on plateau (s)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Post-SFTPRO idle \u2014 {ylabel}")
    ax.legend(fontsize=9)

fig.suptitle("Post-SFTPRO Idle Plateau \u2014 Turn-by-Turn (body)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. LHC Hysteresis

Compare LHC plateau values between sessions.
- **Top** (~5781 A, 7 turns)
- **Injection** (~302 A, brief: ~3 turns)
- **Post-LHC idle** (~155 A) \u2014 control: both sessions descend from same LHC top.

In [ ]:
lhc_plateaus = [
    ("LHC",          "injection",  "LHC injection (~302 A)"),
    ("LHC",          "lhc_top",    "LHC top (~5781 A)"),
    ("post-LHC",     "idle",       "Post-LHC idle (~155 A)"),
]

lhc_results = {}
for cycle, label, desc in lhc_plateaus:
    print(f"\n=== {desc} ===")
    lhc_results[desc] = {}
    for name in SESSION_NAMES:
        g = get_group(all_groups[name], cycle, label)
        if g is None:
            print(f"  {name}: not found")
            continue
        n_settle = N_SETTLE.get(label, min(18, g["n_turns"]))
        sub = get_settled_slice(turns[name], g, n_settle)
        lhc_results[desc][name] = sub
        print(f"  {name}: {g['n_turns']} total, {len(sub)} settled, I={sub.I_A.mean():.1f} A, "
              f"B1={sub.B1_T.mean():.6f} T, b2={sub.b2.mean():.2f}, b3={sub.b3.mean():.2f}")

    if len(lhc_results[desc]) == 2:
        d = lhc_results[desc]["26 GeV"].B1_T.mean() - lhc_results[desc]["200 GeV"].B1_T.mean()
        print(f"  \u0394(26-200) B1 = {d*1e6:.1f} \u00b5T")

In [ ]:
# LHC top turn-by-turn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for col, ylabel, ax in zip(["B1_T", "b2", "b3"],
                            ["B1 (T)", "b2 (units)", "b3 (units)"], axes):
    for name in SESSION_NAMES:
        g = get_group(all_groups[name], "LHC", "lhc_top")
        if g is None:
            continue
        sub = turns[name].iloc[g["start"]:g["end"]]
        local_t = sub.t_s.values - sub.t_s.values[0]
        ax.plot(local_t, sub[col], "o-", ms=4, color=COLORS[name], alpha=0.7, label=name)
    ax.set_xlabel("Time on plateau (s)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"LHC top \u2014 {ylabel}")
    ax.legend(fontsize=9)

fig.suptitle("LHC Top Plateau \u2014 Turn-by-Turn (body)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. NMR at Top Plateaus

NMR (Caylar) locked values at SFTPRO top and LHC top.

Colour code:
- **Grey:** lock = 0 (unlocked)
- **Orange:** lock = 1 but < 1 s since lock acquisition (transient)
- **Green:** lock = 1 and \u2265 1 s of continuous lock before this sample (settled)

In [ ]:
def compute_nmr_colors(ds, settle_s):
    """Classify each NMR sample into unlocked / early-lock / settled-lock."""
    lock = ds["lock"]
    t_s = ds["t_ms"] / 1000.0
    n = len(lock)

    time_since_lock = np.zeros(n)
    lock_start_t = 0.0
    in_lock = False
    for i in range(n):
        if lock[i] == 1:
            if not in_lock:
                in_lock = True
                lock_start_t = t_s[i]
            time_since_lock[i] = t_s[i] - lock_start_t
        else:
            in_lock = False
            time_since_lock[i] = 0.0

    mask_unlocked = lock == 0
    mask_early    = (lock == 1) & (time_since_lock < settle_s)
    mask_settled  = (lock == 1) & (time_since_lock >= settle_s)
    return mask_unlocked, mask_early, mask_settled, t_s


# --- NMR 3-colour plot ---
fig, axes = plt.subplots(len(SESSION_NAMES), 1, figsize=(16, 5 * len(SESSION_NAMES)))
if len(SESSION_NAMES) == 1:
    axes = [axes]

for ax, name in zip(axes, SESSION_NAMES):
    ds = nmr_data[name]
    mask_u, mask_e, mask_s, t_s = compute_nmr_colors(ds, NMR_SETTLE_S)
    nmr_abs = np.abs(ds["nmr"])

    ax.plot(t_s[mask_u], nmr_abs[mask_u], ".", ms=1.5, color="0.70",
            alpha=0.4, label="unlocked", rasterized=True)
    ax.plot(t_s[mask_e], nmr_abs[mask_e], ".", ms=3, color="tab:orange",
            alpha=0.7, label=f"locked < {NMR_SETTLE_S:.0f} s")
    ax.plot(t_s[mask_s], nmr_abs[mask_s], ".", ms=3, color="tab:green",
            alpha=0.8, label=f"locked \u2265 {NMR_SETTLE_S:.0f} s (settled)")

    ax.set_ylabel("|NMR| (T)")
    ax.set_title(f"{name} \u2014 NMR")
    ax.legend(fontsize=9, markerscale=3)

    # Print settled stats
    if mask_s.sum() > 0:
        print(f"{name}: {mask_s.sum()} settled NMR pts, "
              f"|NMR| = {nmr_abs[mask_s].mean():.6f} \u00b1 {nmr_abs[mask_s].std():.6f} T")

axes[-1].set_xlabel("Time (s)")
fig.suptitle("NMR Lock Status \u2014 Grey=unlocked, Orange=early lock, Green=settled",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- NMR zoom on lock periods ---
fig, axes = plt.subplots(len(SESSION_NAMES), 2, figsize=(16, 5 * len(SESSION_NAMES)))

for row, name in enumerate(SESSION_NAMES):
    ds = nmr_data[name]
    mask_u, mask_e, mask_s, t_s = compute_nmr_colors(ds, NMR_SETTLE_S)
    nmr_abs = np.abs(ds["nmr"])
    lock = ds["lock"]

    # Find main lock periods (SFTPRO, LHC)
    diff = np.diff(lock)
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    if lock[-1] == 1:
        ends = np.append(ends, len(lock))

    # Find the two longest lock periods
    durations = [(t_s[min(e-1, len(t_s)-1)] - t_s[s], s, e)
                 for s, e in zip(starts, ends)]
    durations.sort(key=lambda x: -x[0])

    for col_idx, (dur, s, e) in enumerate(durations[:2]):
        ax = axes[row, col_idx]
        t_loc = t_s[s:e] - t_s[s]
        nmr_loc = nmr_abs[s:e]
        tsl_loc = t_loc  # time since lock start

        m_early = tsl_loc < NMR_SETTLE_S
        m_settled = tsl_loc >= NMR_SETTLE_S

        ax.plot(t_loc[m_early], nmr_loc[m_early], ".", ms=4, color="tab:orange",
                alpha=0.7, label=f"< {NMR_SETTLE_S:.0f} s")
        ax.plot(t_loc[m_settled], nmr_loc[m_settled], ".", ms=4, color="tab:green",
                alpha=0.8, label=f"\u2265 {NMR_SETTLE_S:.0f} s")
        ax.axvline(NMR_SETTLE_S, color="grey", ls="--", lw=0.8, alpha=0.5)

        hall_mean = ds["hall"][s:e].mean()
        plat = "LHC top" if hall_mean > HALL_LHC_MIN else "SFTPRO top"
        settled_mean = nmr_loc[m_settled].mean() if m_settled.sum() > 0 else np.nan
        settled_std = nmr_loc[m_settled].std() if m_settled.sum() > 1 else np.nan

        ax.set_title(f"{name} \u2014 {plat} ({dur:.1f} s lock)")
        ax.set_xlabel("Time since lock (s)")
        ax.set_ylabel("|NMR| (T)")
        ax.legend(fontsize=9, markerscale=2)

        if not np.isnan(settled_mean):
            ax.axhline(settled_mean, color="tab:green", ls=":", lw=0.8)
            ax.text(0.98, 0.05, f"settled: {settled_mean:.6f} \u00b1 {settled_std:.6f} T",
                    transform=ax.transAxes, ha="right", fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

fig.suptitle("NMR Lock Periods \u2014 Zoom", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- NMR vs Rotating Coil comparison ---
def get_nmr_settled(name, plat_name):
    ds = nmr_data[name]
    _, _, mask_s, t_s = compute_nmr_colors(ds, NMR_SETTLE_S)
    lock = ds["lock"]
    hall = ds["hall"]
    nmr_abs = np.abs(ds["nmr"])

    diff = np.diff(lock)
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    if lock[-1] == 1:
        ends = np.append(ends, len(lock))

    for s, e in zip(starts, ends):
        dur = t_s[e-1] - t_s[s]
        if dur < 0.5:
            continue
        h = hall[s:e].mean()
        p = "LHC top" if h > HALL_LHC_MIN else "SFTPRO top"
        if p == plat_name:
            settled_mask = (t_s[s:e] - t_s[s]) >= NMR_SETTLE_S
            vals = nmr_abs[s:e][settled_mask]
            if len(vals) > 0:
                return vals.mean(), vals.std(), len(vals)
    return np.nan, np.nan, 0


print("=== NMR vs Rotating Coil Comparison (settled) ===")
print(f"{'Plateau':15s} {'Measure':8s} {'200 GeV':>14s} {'26 GeV':>14s} {'\u0394 (\u00b5T)':>10s}")
print("-" * 65)

for plat_label, cycle, band, nmr_plat in [
    ("SFTPRO top", "SFTPRO", "sftpro_top", "SFTPRO top"),
    ("LHC top",    "LHC",    "lhc_top",    "LHC top"),
]:
    # NMR
    n200, _, _ = get_nmr_settled("200 GeV", nmr_plat)
    n26, _, _  = get_nmr_settled("26 GeV", nmr_plat)
    if not np.isnan(n200):
        print(f"{plat_label:15s} {'NMR':8s} {n200:14.6f} {n26:14.6f} {(n26-n200)*1e6:10.1f}")

    # Rotating coil B1
    rc = {}
    for sname in SESSION_NAMES:
        g = get_group(all_groups[sname], cycle, band)
        if g:
            sub = get_settled_slice(turns[sname], g, N_SETTLE[band])
            rc[sname] = sub.B1_T.mean()
    if len(rc) == 2:
        print(f"{'':15s} {'RC B1':8s} {rc['200 GeV']:14.6f} {rc['26 GeV']:14.6f} "
              f"{(rc['26 GeV']-rc['200 GeV'])*1e6:10.1f}")

---
## 10. Summary

In [ ]:
# Build full comparison table
summary_rows = []

def add_row(plateau, cycle, label):
    r = {"Plateau": plateau}
    for name in SESSION_NAMES:
        g = get_group(all_groups[name], cycle, label)
        if g is None:
            continue
        n_s = N_SETTLE.get(label, min(18, g["n_turns"]))
        sub = get_settled_slice(turns[name], g, n_s)
        r[f"{name} B1 (T)"]     = sub.B1_T.mean()
        r[f"{name} b2 (units)"] = sub.b2.mean()
        r[f"{name} b3 (units)"] = sub.b3.mean()
        r[f"{name} N"]          = len(sub)
    return r

summary_rows.append(add_row("SFTPRO top (4816 A)",        "SFTPRO",       "sftpro_top"))
summary_rows.append(add_row("SFTPRO\u2192LHC idle (155 A)", "SFTPRO\u2192LHC", "idle"))
summary_rows.append(add_row("LHC injection (302 A)",      "LHC",          "injection"))
summary_rows.append(add_row("LHC top (5781 A)",           "LHC",          "lhc_top"))
summary_rows.append(add_row("Post-LHC idle (155 A)",      "post-LHC",     "idle"))

df_sum = pd.DataFrame(summary_rows)

# Add deltas
for qty in ["B1 (T)", "b2 (units)", "b3 (units)"]:
    c200 = f"200 GeV {qty}"
    c26  = f"26 GeV {qty}"
    if c200 in df_sum.columns and c26 in df_sum.columns:
        if "B1" in qty:
            df_sum[f"\u0394 {qty.replace('(T)','(\u00b5T)')}"] = (df_sum[c26] - df_sum[c200]) * 1e6
        else:
            df_sum[f"\u0394 {qty}"] = df_sum[c26] - df_sum[c200]

print("Hysteresis Summary: 26 GeV (flat MD1) minus 200 GeV (full MD1)")
print("=" * 100)
display(df_sum.set_index("Plateau").round(6))

In [ ]:
# Bar chart of deltas
delta_cols = [c for c in df_sum.columns if c.startswith("\u0394")]
if delta_cols:
    fig, axes = plt.subplots(1, len(delta_cols), figsize=(6 * len(delta_cols), 5))
    if len(delta_cols) == 1:
        axes = [axes]
    plateaus = df_sum["Plateau"].values
    x = np.arange(len(plateaus))

    for ax, dc in zip(axes, delta_cols):
        vals = df_sum[dc].values
        colors = ["tab:red" if v > 0 else "tab:blue" for v in vals]
        ax.bar(x, vals, color=colors, alpha=0.7, edgecolor="black", linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(plateaus, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(dc)
        ax.axhline(0, color="grey", lw=0.8)
        ax.set_title(dc)

    fig.suptitle("Hysteresis: \u0394(26 GeV \u2212 200 GeV)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Notes

- **Body segment only** (main magnet body, `is_fringe=False`).
- **Encoder offset correction** applied: raw `arg(C_1) = \u221290.4\u00b0` sits 0.4\u00b0 past
  the `_wrap_arg_to_pm_pi_over_2` boundary at \u2212\u03c0/2. Correction equivalent to
  `encoder_offset_rad = \u2212\u03c0/2`: B1 flips positive, even-order b_n flip sign,
  odd-order and all |b_n| unchanged.
- Per-turn results use `dri rot nor` pipeline from the DAQ software.
- SFTPRO has **no injection plateau** \u2014 ramps directly from idle to top.
- LHC injection is very brief (~3 turns). Statistics are limited.
- NMR values classified: grey=unlocked, orange=locked<1 s, green=locked\u22651 s.
- Post-LHC idle serves as a **control**: both sessions descend from the same
  LHC top (5781 A), so any residual difference indicates long-memory hysteresis.

---
## 11. Session-Ordering Confound Analysis

The 200 GeV session ran **first** (15:22), the 26 GeV session ran **second** (15:35).
This matters because of the **Preisach wiping-out property**:

> Memory of a previous field maximum H_max is only erased when the applied field exceeds H_max again.

**Consequence:** the 26 GeV session inherits the magnetic memory of the 200 GeV session's
LHC excursion to 5781 A. The 20× MD1 cycling to only 301 A is far too weak to erase
this (wiping-out requires exceeding 5781 A). So at SFTPRO (4816 A < 5781 A), the iron
is on a **minor ascending loop** inside the major loop — biased upward by the remanent
magnetization from the descending branch of 5781 A.

The 200 GeV session had no such prior high-field memory, so its ascending branch at
SFTPRO sits lower (closer to the virgin curve).

**Validation:** the post-LHC idle (both sessions descend from the same 5781 A) should
give identical B. A near-zero delta there confirms this interpretation.

In [ ]:
# --- Quantify the confound: signal vs control ---
print("SESSION-ORDERING CONFOUND CHECK")
print("=" * 80)
print("Delta = 26 GeV (ran second) minus 200 GeV (ran first)")
print()

confound_plateaus = [
    ("SFTPRO top (4816 A)",      "SFTPRO",     "sftpro_top",  "SIGNAL"),
    ("LHC top (5781 A)",         "LHC",         "lhc_top",     "SIGNAL"),
    ("Post-SFTPRO idle (155 A)", "SFTPRO\u2192LHC", "idle",    "intermediate"),
    ("Post-LHC idle (155 A)",    "post-LHC",    "idle",        "CONTROL"),
]

print(f"{'Plateau':<30s} {'200 GeV B1 (T)':>16s} {'26 GeV B1 (T)':>16s} "
      f"{'Delta B1 (uT)':>14s} {'Role':>12s}")
print("-" * 92)

for desc, cycle, label, role in confound_plateaus:
    g200 = get_group(all_groups["200 GeV"], cycle, label)
    g26 = get_group(all_groups["26 GeV"], cycle, label)
    if g200 is None or g26 is None:
        print(f"{desc:<30s}  -- missing --")
        continue
    n_s = N_SETTLE.get(label, 18)
    sub200 = get_settled_slice(turns["200 GeV"], g200, n_s)
    sub26 = get_settled_slice(turns["26 GeV"], g26, n_s)
    b200 = sub200.B1_T.mean()
    b26 = sub26.B1_T.mean()
    delta = (b26 - b200) * 1e6
    print(f"{desc:<30s} {b200:>16.6f} {b26:>16.6f} {delta:>+14.1f} {role:>12s}")

print()

# Statistical context
for desc, cycle, label in [("SFTPRO top", "SFTPRO", "sftpro_top"),
                            ("LHC top", "LHC", "lhc_top")]:
    g200 = get_group(all_groups["200 GeV"], cycle, label)
    g26 = get_group(all_groups["26 GeV"], cycle, label)
    n_s = N_SETTLE.get(label, 18)
    sub200 = get_settled_slice(turns["200 GeV"], g200, n_s)
    sub26 = get_settled_slice(turns["26 GeV"], g26, n_s)
    delta = (sub26.B1_T.mean() - sub200.B1_T.mean()) * 1e6
    # Pooled std-of-mean
    sem = np.sqrt(sub200.B1_T.var() / len(sub200) + sub26.B1_T.var() / len(sub26)) * 1e6
    print(f"{desc}: Delta = {delta:+.1f} uT, pooled SEM = {sem:.1f} uT, "
          f"Delta/SEM = {abs(delta)/sem:.1f} sigma")

print()
print("Post-LHC idle control is near zero -> consistent with Preisach:")
print("  the observed signal at SFTPRO/LHC is a session-ordering artifact,")
print("  not a true MD1 conditioning effect.")

---
## 12. Proposed Experiment: Standardized Hysteresis Comparison

The current campaign **conflates** the MD1 conditioning level with the session ordering.
To isolate the true MD1 effect, add **full-field standardization cycles** before each session.

### Protocol

```
SESSION A — 200 GeV MD1:
  10× standardization (0 → 5781 A → 0)     wipes ALL prior memory
  20× MD1 full (0 → 2267 A → 0)            conditioning under test
  idle (155 A) → SFTPRO (4816 A) → idle → inj → LHC (5781 A) → idle

SESSION B — 26 GeV MD1:
  10× standardization (0 → 5781 A → 0)     SAME starting state as A
  20× MD1 flat (0 → 301 A → 0)             different conditioning
  idle (155 A) → SFTPRO (4816 A) → idle → inj → LHC (5781 A) → idle

Then repeat in REVERSE order (B then A) to verify order-independence.
```

### Design principles

1. **10× full-field standardization** before each session ensures identical starting state.
2. **Both session orders** (A→B and B→A) to verify the standardization works.
3. **Longer SFTPRO plateau** if possible (100+ turns → std-of-mean < 83 µT).
4. **NMR as primary instrument** (sub-µT precision). Rotating coil for harmonics.
5. **Post-LHC idle as control** (both sessions descend from 5781 A → should be identical).

### Preisach prediction

With proper standardization, the Preisach model predicts **zero difference** at
SFTPRO and LHC top. Reasoning:

- After standardization, both sessions have memory stack: `(H_max=5781, H_min=0)`
- MD1 conditioning adds `(2267, 0)` or `(301, 0)` to the stack
- When the ascending ramp exceeds the MD1 maximum, the wiping-out property
  erases the MD1 pair → both sessions revert to `(5781, 0)`
- At SFTPRO (4816 A > 2267 A): identical. At LHC top (5781 A): identical.

A **nonzero result** would indicate non-Preisach effects (accommodation, domain-wall
fatigue, thermal) and would be a genuine contribution to understanding SPS dipole
magnetic memory.

### Optional enhancement: intermediate plateau at 2267 A

Add a brief measurement plateau at **2267 A** during the ascending ramp to SFTPRO.
This is the 200 GeV MD1 maximum — the Preisach model predicts the **largest
MD1-dependent difference** right at this current (minor loop rejoins the major
loop here). Above 2267 A, the difference should vanish.

In [ ]:
# --- Visualise the Preisach argument ---
fig, ax = plt.subplots(figsize=(10, 7))

# Schematic B-H curves (illustrative, not from data)
I_range = np.linspace(0, 6000, 500)

# Virgin curve (sigmoid-like)
def schematic_BH(I, B_sat=2.1, I_knee=2000, sharpness=1.5):
    return B_sat * np.tanh(I / I_knee * sharpness)

B_virgin = schematic_BH(I_range)

# Descending from 5781 A (higher B due to remanence)
I_desc = np.linspace(0, 5781, 500)
B_at_5781 = schematic_BH(5781)
B_desc = B_at_5781 - (B_at_5781 - schematic_BH(0) * 0.3) * (1 - I_desc / 5781) ** 0.6

# Minor ascending from descending branch (26 GeV session)
I_minor = np.linspace(0, 5781, 500)
B_remanent = B_desc[0]  # B at I=0 on descending branch
B_minor_asc = B_remanent + (B_at_5781 - B_remanent) * (I_minor / 5781) ** 0.8

# Plot
ax.plot(I_range, B_virgin, "b-", lw=2.5, label="200 GeV session: ~virgin curve (no prior high-field memory)")
ax.plot(I_desc, B_desc, "0.5", lw=1.5, ls="--", label="Descending from LHC top (5781 A) after 200 GeV session")
ax.plot(I_minor, B_minor_asc, "tab:orange", lw=2.5,
        label="26 GeV session: minor ascending (inherits 5781 A memory)")

# Mark key points
for I_mark, label, va in [(4816, "SFTPRO\n4816 A", "bottom"), (5781, "LHC top\n5781 A", "bottom")]:
    ax.axvline(I_mark, color="grey", ls=":", lw=0.8, alpha=0.5)
    ax.text(I_mark, 0.05, label, ha="center", va=va, fontsize=8, color="grey")

# Mark MD1 levels
for I_md1, label, color in [(2267, "200 GeV MD1\nmax (2267 A)", "tab:blue"),
                              (301, "26 GeV MD1\nmax (301 A)", "tab:orange")]:
    ax.axvline(I_md1, color=color, ls="--", lw=0.8, alpha=0.4)
    ax.text(I_md1 + 50, B_at_5781 * 0.15, label, fontsize=7, color=color, rotation=90, va="bottom")

# Mark the gap at SFTPRO
idx_sft = np.argmin(np.abs(I_range - 4816))
idx_sft_minor = np.argmin(np.abs(I_minor - 4816))
B_virgin_sft = B_virgin[idx_sft]
B_minor_sft = B_minor_asc[idx_sft_minor]
ax.annotate("", xy=(4950, B_minor_sft), xytext=(4950, B_virgin_sft),
            arrowprops=dict(arrowstyle="<->", color="red", lw=1.5))
ax.text(5050, (B_minor_sft + B_virgin_sft) / 2, f"\u0394B \u2248 125 \u00b5T",
        fontsize=10, color="red", va="center", fontweight="bold")

ax.set_xlabel("I (A)", fontsize=12)
ax.set_ylabel("B (T)  [schematic]", fontsize=12)
ax.set_title("Session-Ordering Confound: Preisach Interpretation (schematic)", fontsize=13)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.set_xlim(-100, 6300)
ax.set_ylim(0, B_at_5781 * 1.1)

plt.tight_layout()
plt.show()

print("NOTE: This is a schematic illustration. The curves are qualitative,")
print("not fitted to data. The key point is the vertical gap at SFTPRO")
print("between the two ascending branches.")